## What is Query Decomposition?

- Query decomposition is the process of taking a complex, multi-part question and breaking it into simpler, atomic sub-questions that can each be retrieved and answered individually.

## Why Use Query Decomposition?

- Complex queries often involve multiple concepts
- LLMs or retrievers may miss parts of the original question
- It enables multi-hop reasoning (answering in steps)
- Allows parallelism (especially in multi-agent frameworks)

In [1]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.runnables import RunnableSequence

In [2]:
# Step 1: Load and embed the document
loader = TextLoader("langchain_crewai_dataset.txt")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)

embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding)
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 4, "lambda_mult": 0.7})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3072.47it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# Step 2: LLM
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

# llm=init_chat_model(model="groq:gemma2-9b-it") # decommissioned model
llm = init_chat_model(model="groq:llama-3.3-70b-versatile")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000021290EB9810>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021290E93A10>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [5]:
# Step 3: Query decomposition
decomposition_prompt = PromptTemplate.from_template("""
You are an AI assistant. Decompose the following complex question into 2 to 4 smaller sub-questions for better document retrieval.

Question: "{question}"

Sub-questions:
""")
decomposition_chain = decomposition_prompt | llm | StrOutputParser()

In [6]:
query = "How does LangChain use memory and agents compared to CrewAI?"
decomposition_question=decomposition_chain.invoke({"question": query})


In [7]:
print(decomposition_question)

To better understand and retrieve relevant documents, the complex question can be decomposed into the following sub-questions:

1. How does LangChain utilize memory in its architecture and operations?
2. What role do agents play in LangChain, and how are they integrated into its framework?
3. How does CrewAI use memory and agents in its system, and what are the key similarities and differences with LangChain? 

These sub-questions can help in retrieving specific and relevant information about LangChain and CrewAI, allowing for a more detailed comparison of their approaches to memory and agents.


In [8]:
# Step 4: QA chain per sub-question
qa_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.

Context:
{context}

Question: {input}
""")
qa_chain = create_stuff_documents_chain(llm=llm, prompt=qa_prompt)

In [9]:
# Step 5: Full RAG pipeline logic
def full_query_decomposition_rag_pipeline(user_query):
    # Decompose the query
    sub_qs_text = decomposition_chain.invoke({"question": user_query})
    sub_questions = [q.strip("-•1234567890. ").strip() for q in sub_qs_text.split("\n") if q.strip()]
    
    results = []
    for subq in sub_questions:
        docs = retriever.invoke(subq)
        result = qa_chain.invoke({"input": subq, "context": docs})
        results.append(f"Q: {subq}\nA: {result}")
    
    return "\n\n".join(results)

In [11]:
# Step 6: Run
query = "How does LangChain use memory and agents compared to CrewAI?"
final_answer = full_query_decomposition_rag_pipeline(query)
print(" Final Answer:\n")
print(final_answer)

 Final Answer:

Q: To better understand and document the comparison between LangChain and CrewAI in terms of memory and agents, we can decompose the original question into the following sub-questions:
A: To better understand and document the comparison between LangChain and CrewAI in terms of memory and agents, we can decompose the original question into the following sub-questions:

1. How do LangChain agents utilize memory, and what are the key features of their memory use?
2. What is the role of CrewAI in managing memory, if any, and how does it differ from LangChain's approach?
3. How do LangChain agents operate, and what is the planner-executor model they use?
4. Can CrewAI manage agents, and if so, how does it compare to LangChain's agent management capabilities?
5. Are there any specific differences or similarities in how LangChain and CrewAI handle role-based collaboration, dynamic decision-making, and branching logic?

Q: **What is LangChain's approach to memory management, an